# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [6]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import re
import pandas as pd

#https://realpython.com/beautiful-soup-web-scraper-python/

def get_urls(page_url, urls=None, not_correct_urls=None):
    headers = { #We need this to be able to have access pretty much to get the data.
        "User-Agent": (
            "Chrome/124.0.0.0 "
            "Edg/150.0.4078.48"
        )
    }

    content = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(content.content, "html.parser") #uses html parser for the response because it is html and create the BeautifulSoup object with content as content as input.

    if not urls: #I create a new set if it does not exist
        urls = set()
    if not not_correct_urls: #I create a new set if it does not exist
        not_correct_urls = set()

    for a in soup.find_all("a", href=True): #I get all the a tags
        href = a["href"]
        href = re.sub(r"^(?:\.\./)+", "/", href) #The paths are like ../../../ so I am replacing them with a / so I can add the actual url
        href = "https://www.fixr.com" + href
        # Fixr cost pages always contain /costs/ , so we are looking for these

        if "/costs/" in href: #If this exists than we want to add it unless it already exists. i know that it is a set and duplicates cannot exist but I want to be careful
            if href not in urls:
                urls.add(href)
        else: #if not, add the href to the not correct urls set.
            if href not in not_correct_urls:
                not_correct_urls.add(href)
    return urls, not_correct_urls

In [7]:
def scrape_data(url):
    headers = {
        "User-Agent": (
            "Chrome/124.0.0.0 "
            "Edg/150.0.4078.48"
        )
    }
    #doing same thing
    content = requests.get(url, headers=headers).text
    soup = BeautifulSoup(content, "html.parser")

    data = { #dict containing the description and url
        "url": url,
        "description": None,
    }

    # Extract average cost
    for p in soup.find_all("p"): #the description is in the p tag. In the description, it has text containing the national average cost
        #only if the p has "average" in the text
        clean_text = p.get_text(strip=True).lower()
        if "average" in clean_text:
            #they could be using average but talking about something else, so we check that the dollar sign is in the text.
            if "$" in clean_text:
                data["description"] = clean_text #we add it to the dict.
                break #break out of loop

    return data


In [8]:
url_set, not_in_set = get_urls("https://www.fixr.com/costs/")

for url in not_in_set.copy(): #I got a error when I didn't use a copy of the not_in_set because I am updating it during the loop. so I am using copy.
    new_set, new_not_in_set = get_urls(url, url_set, not_in_set) # I get urls from the not_in_set
    for u in new_set:
        if u not in url_set:
            url_set.add(u) #I add to url set if it is in new set. (this will inlcude that past urls already in url_set, so we use this condition just in case even thought it is a set)

    for u in new_not_in_set:
        not_in_set.add(u) #I add to not_in_set set if it is in new set. (this will inlcude that past urls already in not_in_set, so we use this condition just in case even thought it is a set)

urls = dict() #we initialize an empty dict
for url in url_set: #I am getting the last part of the https:____/__/name
   name = url.split("/")[-1]
   urls[name] = url #I add this to my dict with the name as key and url as value

dataset = {name: scrape_data(url) for name, url in urls.items()} #for each url, I am scraping the data from it and soring it in this dict as my dataset..

In [41]:
rows = []
for name, data in dataset.items(): #name is the key and data is the value.
    row = { #I am looping through this and setting up a dict for creating the dataframe
        "item": name,
        "url": data["url"],
        "description": data["description"]
    }
    rows.append(row) #I append to the rows list

df = pd.DataFrame(rows) #I create the dataframe from the rows
df["description"] = df["description"].str.strip().str.lower() #making sure the description is cleaned.

df["positive"] = np.zeros(df["description"].shape).astype(np.uint8) #I am adding categories and filling them with 0 to start
df["negative"] = np.zeros(df["description"].shape).astype(np.uint8)
df["neutral"] = np.zeros(df["description"].shape).astype(np.uint8)


In [42]:
avg_cost_list = []
na_idx = []

for i, row in enumerate(df["description"]):
    if row:
        row = row.replace(",", "") #remove commas so it is easy when getting the money parts
        # https://www.rexegg.com/regex-quantifiers.php
        #I am using this regex pattern to find the value after average.
        avg_val = re.findall(r"average.*?\$(\d+)", row) #average and \*? (this is called a lazy modifier). It is trying to get the quickest match pretty much. the mach looks for $. it can include digits. I surround it by ( ) for capturing the digits
        avg_val = np.array(avg_val).astype(np.long) #I make it a numpy array

        #In this section, I am checking the shape because some of the websites information is different and doesn't include average values. I only want ones that match the type of shape I want.
        if avg_val.shape != (1,):
            na_idx.append(i)
        else:
            avg_cost_list.append(float(avg_val[0]))
    else:
        na_idx.append(i)

df = df.drop(index=na_idx).reset_index(drop=True)
df["avg_cost"] = avg_cost_list

In [43]:
df.loc[:, "item"] = df["item"].str.replace(r"(?:-|#)", " ", regex=True) #removing special characters with space. before: item-replacement | after: item replacement

index = df[df["item"].str.contains("roof replacement ")].index #removes this because it includes specific state values. I only need the national average.
df = df.drop(index=index).reset_index(drop=True) #drop it and reset the index

index = df[df["item"].astype(str).str.contains("solar panel installation ")]["item"].index #removes this because it includes specific state values. I only need the national average.
df = df.drop(index=index).reset_index(drop=True)

index = df[(df["avg_cost"] > 50000) & ~(df["item"].str.contains(r"pool"))].index #This data had things that were above 50000 like building a big thing. I excluded it except for adding a pool
df = df.drop(index=index).reset_index(drop=True)

index = df[df["avg_cost"] < 100].index #This data had stuff that were hourly wages or stuff in the 6 figure numbers as like one number, so I am dropping them.
df = df.drop(index=index).reset_index(drop=True)

index = df[df["item"].astype(str).str.contains("treatment ")].index #This is used for termite treatments. I use space because only states are added afterwards on certain ones.
df = df.drop(index=index).reset_index(drop=True)

index = df[df["item"].str.contains("sqft roof")].index #I am dropping these specifics to make the data more simple as this is a project. their is a average roof replacement cost that I want to be used instead.
df = df.drop(index=index).reset_index(drop=True)

In [44]:
#I create patterns to look for. This data has pests which are negative and other negative stuff I included in this.
negative_words_pattern = re.compile(r"\b(removal|replace|exterm|pest|damage|leak|crack|rot|mold|mildew|flood|burst|slab|unclog|demoli|restor|remediat|odor|termite|rat|mouse|squirrel|raccoon|skunk|bat|bee|wasp|hornet|spider|ant|centipede|groundhog|gopher|vole|mole|snake|cicada|bug|beetle|hornets)\w*\b")
# \b marks the start of the word and end of the word. \w represents any word character. * means zero or more. by doing this, I can put install and any variations will also count like installation.
positive_words_pattern = re.compile(r"\b(install|add|new|upgrad|renovat|remodel|finish|modern|update|design|build|refac|refinish|luxury|custom|solar panel|pool|sauna|home theater|home gym|wine cellar|vault|mudroom|pergola|gazebo|sunroom|greenhouse|treehouse|landscape|smart)\w*\b") #it is hard for me to determine what should go in here. I feel like install and add could be considered neutral but based on the type of item names, I am keeping it as positive.
#


In [45]:
for idx, word in enumerate(df["item"]): #I pretty much check whether something matches the patterns
    negative_check = False
    positive_check = False
    neutral_check = False

    if negative_words_pattern.search(word):
        negative_check = True
    if positive_words_pattern.search(word):
        positive_check = True
    #if both checks are true or false than I will set neutral check to true.
    if (negative_check and positive_check) or (not negative_check and not positive_check):
        neutral_check = True

    if neutral_check: #I do this first
        df.loc[idx, "neutral"] = 1
    elif negative_check: #check negative next
        df.loc[idx, "negative"] = 1
    else:
        df.loc[idx, "positive"] = 1

#I change the type to category. It is as if I did one hot encoding by hand. not really though.
df["positive"] = df["positive"].astype("category")
df["negative"] = df["negative"].astype("category")
df["neutral"] = df["neutral"].astype("category")

In [46]:
df[["positive", "negative", "neutral"]].value_counts()

,,,count
positive,negative,neutral,
0,0,1,336
1,0,0,268
0,1,0,88


In [47]:
cleaned_df = df.drop(columns=["url", "description"]) #I am getting only the columns I want
cleaned_df.to_csv("cleaned_house_repair_dataset.csv", index=False) #I am creating csv file with values